# Notebook 01 Dataset Preparation
**FinAlign | Parts A (Sharvari) + B (Samiksha)**

End-to-end data pipeline:  
Part A: Load HuggingFace data ? Length filter ? Deduplication ? PII scrub ? Format standardise  
Part B: Quality scoring ? Stratified split ? 500 preference pairs ? 100-question eval set

## 0. Install

In [ ]:
!pip install -q datasets huggingface_hub
import sys; sys.path.insert(0, 'src')
print("Ready")

## PART A Sharvari

### A1. Load Raw Data from HuggingFace

In [ ]:
from data_pipeline import load_huggingface_data
raw_records = load_huggingface_data(cache_dir="data/raw")
print(f"Total raw records loaded: {len(raw_records)}")

# Preview
for r in raw_records[:2]:
    print("---")
    print("INSTRUCTION:", r["instruction"][:120])
    print("RESPONSE:   ", r["response"][:120])
    print("SOURCE:     ", r["source"])

### A2. Length Filter

In [ ]:
from data_pipeline import length_filter
filtered, f_stats = length_filter(
    raw_records,
    min_instruction_chars=20,
    max_instruction_chars=2000,
    min_response_chars=100,
    max_response_chars=8000,
    min_response_words=20,
)
print("\nLength Filter Stats:")
for k, v in f_stats.items():
    print(f"  {k}: {v}")

### A3. Deduplication (Exact + Near-duplicate)

In [ ]:
from data_pipeline import deduplicate
deduped, d_stats = deduplicate(filtered, jaccard_threshold=0.7)
print("\nDedup Stats:")
for k, v in d_stats.items():
    print(f"  {k}: {v}")

### A4. PII Scrubbing

In [ ]:
from data_pipeline import scrub_pii
scrubbed, pii_stats = scrub_pii(deduped)
print("\nPII Scrub Stats:", pii_stats)

# Show a before/after example if any PII was found
for orig, clean in zip(deduped[:500], scrubbed[:500]):
    if orig["instruction"] != clean["instruction"] or orig["response"] != clean["response"]:
        print("\nBEFORE:", orig["instruction"][:200])
        print("AFTER: ", clean["instruction"][:200])
        break

### A5. Format Standardize ? {instruction, input, output}

In [ ]:
from data_pipeline import standardize_format
final_a, fmt_stats = standardize_format(scrubbed)
print("\nFormat Stats:")
for k, v in fmt_stats.items():
    print(f"  {k}: {v}")
print("\nSample standardized record:")
import json
print(json.dumps(final_a[0], indent=2))

### A6. Save Cleaned Dataset

In [ ]:
import json
from pathlib import Path
Path("data/processed").mkdir(parents=True, exist_ok=True)
out = "data/processed/finance_sft_clean.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for r in final_a:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(final_a)} records -> {out}")
assert len(final_a) >= 5000, f"Need 5000+ pairs, got {len(final_a)}"
print("Target 5,000+ pairs: PASSED")

---
## PART B Samiksha

### B1. Quality Scoring

In [ ]:
from data_pipeline import quality_score
import random

sample = random.sample(final_a, min(1000, len(final_a)))
scores = [quality_score(r) for r in sample]
totals = [s["total"] for s in scores]

import statistics
print(f"Quality Score Distribution (0-9 scale, sample n={len(scores)}):")
print(f"  Mean  : {statistics.mean(totals):.2f}")
print(f"  Median: {statistics.median(totals):.2f}")
print(f"  Stdev : {statistics.stdev(totals):.2f}")
print(f"  Min   : {min(totals)}")
print(f"  Max   : {max(totals)}")
from collections import Counter
dist = Counter(totals)
for score in sorted(dist):
    bar = "#" * (dist[score] // 5)
    print(f"  {score}: {bar} ({dist[score]})")

### B2. Stratified Train/Val/Test Split

In [ ]:
from data_pipeline import stratified_split
train, val, test, split_stats = stratified_split(
    final_a, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10,
    quality_threshold=4, seed=42,
)

print("\nSplit Stats:")
print(f"  Train : {len(train)}")
print(f"  Val   : {len(val)}")
print(f"  Test  : {len(test)}")
print("\nTopic breakdown (train):")
from collections import Counter
train_topics = Counter(r.get("_topic","?") for r in train)
for t, c in sorted(train_topics.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c}")

### B3. Generate 500 Preference Pairs

In [ ]:
from data_pipeline import generate_preference_pairs
pairs = generate_preference_pairs(train, n_pairs=500, seed=42, topic_balance=True)

print(f"\nGenerated: {len(pairs)} preference pairs")
print("\n--- Sample Pairs (sanity check) ---")
import random
samples = random.sample(pairs, min(5, len(pairs)))
for i, p in enumerate(samples, 1):
    print(f"\n[Pair {i}] Topic: {p['topic']}")
    print(f"  PROMPT:   {p['prompt'][10:100]}...")
    print(f"  CHOSEN:   {p['chosen'][:150]}")
    print(f"  REJECTED: {p['rejected'][:150]}")

### B4. Save Splits + Preference Pairs

In [ ]:
from pathlib import Path
import json

Path("data/preference_pairs").mkdir(parents=True, exist_ok=True)

def _strip(rs):
    return [{k:v for k,v in r.items() if not k.startswith("_")} for r in rs]

for name, records in [("train", train), ("val", val), ("test", test)]:
    path = f"data/processed/{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for r in _strip(records):
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(records)} records)")

n_tr = int(len(pairs) * 0.8)
for name, ps in [("train_pref", pairs[:n_tr]), ("val_pref", pairs[n_tr:])]:
    path = f"data/preference_pairs/{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for p in ps:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(ps)} pairs)")

### B5. Generate 100-Question Eval Set

In [ ]:
from data_pipeline import generate_eval_set
from pathlib import Path
import json

eval_items = generate_eval_set(test, n_questions=100, seed=42)
Path("data/eval_set").mkdir(parents=True, exist_ok=True)
with open("data/eval_set/benchmark_100q.jsonl", "w", encoding="utf-8") as f:
    for item in eval_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"Eval set saved: {len(eval_items)} questions")
from collections import Counter
print("Topic distribution:", dict(Counter(e["topic"] for e in eval_items)))

### B6. Leakage Check

In [ ]:
# Verify no prompt leakage between train and val/test
train_instr = {r["instruction"].strip().lower() for r in train}
val_instr   = {r["instruction"].strip().lower() for r in val}
test_instr  = {r["instruction"].strip().lower() for r in test}

train_val_leak  = len(train_instr & val_instr)
train_test_leak = len(train_instr & test_instr)

print(f"Train-Val overlap  : {train_val_leak}  {'PASS' if train_val_leak == 0 else 'FAIL'}")
print(f"Train-Test overlap : {train_test_leak}  {'PASS' if train_test_leak == 0 else 'FAIL'}")

### Data Stats Report

In [ ]:
import json

stats = {
    "part_a": {
        "raw_count":           len(raw_records),
        "after_length_filter": len(filtered),
        "after_dedup":         len(deduped),
        "after_pii_scrub":     len(scrubbed),
        "after_format_topic":  len(final_a),
        "pct_removed":         round(100*(len(raw_records)-len(final_a))/max(1,len(raw_records)),2),
        "length_filter_breakdown": f_stats["breakdown"],
        "pii_replacements":    pii_stats,
    },
    "part_b": {
        "train": len(train), "val": len(val), "test": len(test),
        "train_pref_pairs": n_tr,
        "val_pref_pairs":   len(pairs) - n_tr,
        "eval_questions":   len(eval_items),
        "topic_breakdown":  {t:c for t,c in split_stats["topic_breakdown"].items()},
    }
}
with open("data/data_stats_report.json", "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))